In [1]:
import os

In [ ]:
%pwd

In [2]:
os.chdir('../')

In [ ]:
%pwd

In [3]:
from pydantic import BaseModel, ConfigDict, Field
from pathlib import Path

class PrepareBaseModelConfig(BaseModel):
    model_config = ConfigDict(extra='forbid')

    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path


class Params(BaseModel):
    model_config = ConfigDict(extra='ignore', populate_by_name=True)

    image_size: list[int] = Field(..., alias='IMAGE_SIZE')
    learning_rate: float = Field(..., alias='LEARNING_RATE')
    include_top: bool = Field(..., alias='INCLUDE_TOP')
    weights: str = Field(..., alias='WEIGHTS')
    classes: int = Field(..., alias='CLASSES')


class ParamsConfig(BaseModel):
    model_config = ConfigDict(extra='forbid')

    prepare_base_model: PrepareBaseModelConfig
    params: Params

In [4]:
from pydantic import BaseModel, ConfigDict
from pathlib import Path

class DataIngestionConfig(BaseModel):
    model_config = ConfigDict(extra='forbid')

    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path


class Config(BaseModel):
    model_config = ConfigDict(extra='forbid')

    artifacts_root: Path
    data_ingestion: DataIngestionConfig
    prepare_base_model: PrepareBaseModelConfig

In [5]:
from src.end_to_end_deep_learning_project_yt.constants import constants
from src.end_to_end_deep_learning_project_yt.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = constants.CONFIG_FILE_PATH,
        params_filepath = constants.PARAMS_FILE_PATH
    ) -> None:
        self.config = read_yaml(config_filepath, Config)
        self.params = read_yaml(params_filepath, Params)

        create_directories([self.config.artifacts_root])


    def get_params_config(self) -> ParamsConfig:
        config = self.config.prepare_base_model
        params = self.params

        create_directories([config.root_dir])

        params_config = ParamsConfig(
            prepare_base_model = config
            , params = params
        )

        return params_config

In [7]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf

I0000 00:00:1794057852.287976   46517 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [8]:
class PrepareBaseModel:
    def __init__(self, config: ParamsConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.applications.vgg16.VGG16(
            input_shape = self.config.params.image_size,
            weights = self.config.params.weights,
            include_top = self.config.params.include_top
        )

        self.save_model(
            path=self.config.prepare_base_model.base_model_path
            , model=self.model
        )

    @staticmethod
    def _prepare_full_model(model, classes, freeze_all, freeze_till, learning_rate):
        if freeze_all:
            for layer in model.layers:
                layer.trainable = False
        elif (freeze_till is not None) and (freeze_till > 0):
            for layer in model.layers[:-freeze_till]:
                model.trainable = False

        flatten_in = tf.keras.layers.Flatten()(model.output)
        prediction = tf.keras.layers.Dense(
            units=classes,
            activation='softmax'
        )(flatten_in)

        full_model = tf.keras.models.Model(
            inputs=model.input,
            outputs=prediction
        )

        full_model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
            loss = tf.keras.losses.CategoricalCrossentropy(),
            metrics=['accuracy']
        )

        full_model.summary()
        return full_model

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params.classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params.learning_rate
        )

        self.save_model(
            path=self.config.prepare_base_model.updated_base_model_path
            , model=self.full_model
        )

In [9]:
try:
    config = ConfigurationManager()
    prepare_base_model_config = config.get_params_config()
    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e:
    raise e

[2026-11-07 13:24:13,981]: INFO - common - yaml file: config/config.yaml loaded successfully
[2026-11-07 13:24:13,983]: INFO - common - yaml file: params.yaml loaded successfully


I0000 00:00:1794057855.774691   46517 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2153 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │        50,178 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,764,866 (56.32 MB)

 Trainable params: 50,178 (196.01 KB)

 Non-trainable params: 14,714,688 (56.13 MB)